# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
%load_ext dotenv
%dotenv 

import os
import sys
sys.path.append(os.getenv('SRC_DIR'))
import numpy as np
import pandas as pd

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
# Assign target variable as the reported burned area of the forest
target = fires_dt['area']
target

0       0.00
1       0.00
2       0.00
3       0.00
4       0.00
       ...  
512     6.44
513    54.29
514    11.16
515     0.00
516     0.00
Name: area, Length: 517, dtype: float64

In [4]:
# Select feature variables (all columns but target column area). Note that most are numerical variables, except for categorical variables month and day
features_df =  fires_dt.drop('area', axis = 1)
features_df

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
512,4,3,aug,sun,81.6,56.7,665.6,1.9,27.8,32,2.7,0.0
513,2,4,aug,sun,81.6,56.7,665.6,1.9,21.9,71,5.8,0.0
514,7,4,aug,sun,81.6,56.7,665.6,1.9,21.2,70,6.7,0.0
515,1,4,aug,sat,94.4,146.0,614.7,11.3,25.6,42,4.0,0.0


In [5]:
# Check for null values
fires_dt.isnull().sum()

coord_x    0
coord_y    0
month      0
day        0
ffmc       0
dmc        0
dc         0
isi        0
temp       0
rh         0
wind       0
rain       0
area       0
dtype: int64

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [6]:
from sklearn.preprocessing import StandardScaler # Preprocessing 1 with standard scaler for numeric variables 
from sklearn.preprocessing import OneHotEncoder # One-hot encoding for categorical values month and day
from sklearn.compose import ColumnTransformer

# Use column transformer to combine scaling and variable encoding
preproc1_transformer = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), features_df.drop(['month','day'],axis=1).columns),
        ('oh_encoder', OneHotEncoder(handle_unknown='infrequent_if_exist'), features_df[['month','day']].columns)
    ], remainder='drop'
)

preproc1_transformer

ColumnTransformer(transformers=[('num_scaler', StandardScaler(),
                                 Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                ('oh_encoder',
                                 OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                 Index(['month', 'day'], dtype='object'))])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [7]:
# Preprocessing 2 with PowerTransformer, yeo-johnson method (non-linear transformation) applied to all numeric variables
# One-hot encoding for categorical values month and day

from sklearn.preprocessing import PowerTransformer
PT_scaler = PowerTransformer()

preproc2_transformer = ColumnTransformer(
    transformers=[('num_scaler2', PowerTransformer(method='yeo-johnson'), features_df.drop(['month','day'],axis=1).columns),
                     ('oh_encoder', OneHotEncoder(handle_unknown='infrequent_if_exist'), features_df[['month','day']].columns)
                     ], remainder='drop'
)

preproc2_transformer


ColumnTransformer(transformers=[('num_scaler2', PowerTransformer(),
                                 Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                ('oh_encoder',
                                 OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                 Index(['month', 'day'], dtype='object'))])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [8]:
# # Pipeline A = preproc1 (StandardScaler + OneHotEncoder) + baseline regressor Lasso
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso

preprocessing1 = preproc1_transformer
regressor1 = Lasso()

pipe_simple1 = Pipeline([
    ('preprocess', preprocessing1),
    ('model', regressor1)
])
pipe_simple1

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num_scaler',
                                                  StandardScaler(),
                                                  Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                                 ('oh_encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  Index(['month', 'day'], dtype='object'))])),
                ('model', Lasso())])

In [9]:
# Pipeline B = preproc2 (PowerTransformer, OneHotEncoder) + baseline regressor Lasso
preprocessing2 = preproc2_transformer
regressor1 = Lasso()

pipe_simple2 = Pipeline([
    ('preprocess', preprocessing2),
    ('model', regressor1)
])

pipe_simple2

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num_scaler2',
                                                  PowerTransformer(),
                                                  Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                                 ('oh_encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  Index(['month', 'day'], dtype='object'))])),
                ('model', Lasso())])

In [10]:
# Pipeline C = preproc1 (StandardScaler + OneHotEncoder) + advanced model DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor

preprocessing1 = preproc1_transformer
regressor2 = DecisionTreeRegressor()

pipe_adv1 = Pipeline([
    ('preprocess', preprocessing1),
    ('model', regressor2)
])

pipe_adv1

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num_scaler',
                                                  StandardScaler(),
                                                  Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                                 ('oh_encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  Index(['month', 'day'], dtype='object'))])),
                ('model', DecisionTreeRegressor())])

In [11]:
# Pipeline D = preproc2 (PowerTransformer + OneHotEncoder) + advanced model DecisionTreeRegressor

preprocessing2 = preproc2_transformer
regressor2 = DecisionTreeRegressor()

pipe_adv2 = Pipeline([
    ('preprocess', preprocessing2),
    ('model', regressor2)
])

pipe_adv2

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num_scaler2',
                                                  PowerTransformer(),
                                                  Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                                 ('oh_encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                  Index(['month', 'day'], dtype='object'))])),
                ('model', DecisionTreeRegressor())])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [12]:
pipe_simple1.get_params() # Check which hyperparameter to tune for Lasso: use alpha

{'memory': None,
 'steps': [('preprocess',
   ColumnTransformer(transformers=[('num_scaler', StandardScaler(),
                                    Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
          'rain'],
         dtype='object')),
                                   ('oh_encoder',
                                    OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                    Index(['month', 'day'], dtype='object'))])),
  ('model', Lasso())],
 'verbose': False,
 'preprocess': ColumnTransformer(transformers=[('num_scaler', StandardScaler(),
                                  Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
        'rain'],
       dtype='object')),
                                 ('oh_encoder',
                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                  Index(['month', 'day'], dtype='object'))]),
 'model': Lasso(),

In [13]:
pipe_adv1.get_params() # Check which hyperparameter to tune for DecisionTreeRegressor: use max_depth

{'memory': None,
 'steps': [('preprocess',
   ColumnTransformer(transformers=[('num_scaler', StandardScaler(),
                                    Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
          'rain'],
         dtype='object')),
                                   ('oh_encoder',
                                    OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                    Index(['month', 'day'], dtype='object'))])),
  ('model', DecisionTreeRegressor())],
 'verbose': False,
 'preprocess': ColumnTransformer(transformers=[('num_scaler', StandardScaler(),
                                  Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
        'rain'],
       dtype='object')),
                                 ('oh_encoder',
                                  OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                  Index(['month', 'day'], dtype='object'))]),
 '

In [14]:
# Set up train-test split
from sklearn.model_selection import train_test_split, GridSearchCV

X = features_df
Y = fires_dt['area']

scoring = 'neg_root_mean_squared_error'

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

# Set up parameter grids for baseline and advanced regressors: Lasso and DecisionTreeRegressor
param_grid_Lasso = {
    'model__alpha': [0.01, 1.0, 2.0, 3.0, 4.0, 5.0]
    }

param_grid_DecisionTreeRegressor = {
    'model__max_depth': [ 1, 2, 4, 6, 8, 10]
    }

In [15]:
# Pipeline 1: pipe_simple 1 (preproc1 and Lasso)
grid_cv1 = GridSearchCV(
    estimator=pipe_simple1, 
    param_grid=param_grid_Lasso, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_root_mean_squared_error")

grid_cv1.fit(X_train, Y_train) # Fit using training dataset

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num_scaler',
                                                                         StandardScaler(),
                                                                         Index(['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind',
       'rain'],
      dtype='object')),
                                                                        ('oh_encoder',
                                                                         OneHotEncoder(handle_unknown='infrequent_if_exist'),
                                                                         Index(['month', 'day'], dtype='object'))])),
                                       ('model', Lasso())]),
             param_grid={'model__alpha': [0.01, 1.0, 2.0, 3.0, 4.0, 5.0]},
             refit='neg_root_mean_squared_error',
             scoring='neg_root_mean_squared_error')

In [16]:
# Show GridSearch results for pipe_simple1 with cv = 5
res1 = grid_cv1.cv_results_
res1 = pd.DataFrame(res1)
res1 = res1.assign(experiment = 1)

# Subset to show columns needed for model evaluation
res1[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_model__alpha', 'split0_test_score',
       'split1_test_score', 'split2_test_score', 'split3_test_score',
       'split4_test_score', 'mean_test_score',
       'std_test_score', 'rank_test_score', 'experiment']].sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,experiment
3,0.011679,0.000607,0.006916,0.000349,3.0,-40.034376,-15.776269,-26.679832,-85.274513,-24.255361,-38.404070,24.694766,1,1
2,0.011955,0.000389,0.006693,0.000542,2.0,-39.895497,-16.164909,-26.916502,-85.005660,-24.120942,-38.420702,24.513194,2,1
4,0.012457,0.000626,0.006631,0.000613,4.0,-40.225119,-15.607612,-26.635726,-85.331419,-24.486793,-38.457334,24.727884,3,1
5,0.051256,0.049459,0.021737,0.021343,5.0,-40.354023,-15.598514,-26.749465,-85.331419,-24.634529,-38.533590,24.703919,4,1
1,0.014214,0.002303,0.007270,0.001419,1.0,-39.863493,-16.857978,-28.499469,-84.745002,-24.290635,-38.851315,24.125240,5,1
0,0.070787,0.073421,0.011931,0.004131,0.01,-40.792974,-19.471543,-33.434988,-85.238861,-26.799629,-41.147599,23.148932,6,1


In [17]:
# Pipeline 2: pipe_simple 2 (preproc2 and Lasso Regressor)
grid_cv2 = GridSearchCV(
    estimator=pipe_simple2, 
    param_grid=param_grid_Lasso, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_root_mean_squared_error")

grid_cv2.fit(X_train, Y_train) # Fit using training dataset

# Show GridSearch results for pipe_simple2 with cv = 5
res2 = grid_cv2.cv_results_
res2 = pd.DataFrame(res2)
res2 = res2.assign(experiment = 2)

# Subset to show columns needed for model evaluation
res2[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_model__alpha', 'split0_test_score',
       'split1_test_score', 'split2_test_score', 'split3_test_score',
       'split4_test_score', 'mean_test_score',      
       'std_test_score', 'rank_test_score', 'experiment']].sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,experiment
3,0.066428,0.032875,0.008570,0.001546,3.0,-40.045972,-15.880708,-26.675575,-85.297687,-24.286884,-38.437365,24.681394,1,2
2,0.150012,0.063112,0.013790,0.004533,2.0,-39.900119,-16.276551,-26.969966,-85.044311,-24.133176,-38.464825,24.501249,2,2
4,0.052998,0.010763,0.008576,0.001260,4.0,-40.253091,-15.709806,-26.648680,-85.331419,-24.534045,-38.495408,24.702833,3,2
5,0.055526,0.004977,0.009044,0.001519,5.0,-40.375074,-15.684353,-26.770889,-85.331419,-24.651410,-38.562629,24.684357,4,2
1,0.048294,0.006519,0.010993,0.006338,1.0,-39.813881,-17.005313,-27.600912,-84.781534,-24.280995,-38.696527,24.192996,5,2
0,0.063578,0.015841,0.008382,0.000630,0.01,-40.556610,-19.623599,-29.206803,-85.075689,-26.713925,-40.235325,23.409662,6,2


In [18]:
# Pipeline 3: pipe_adv1
grid_cv3 = GridSearchCV(
    estimator=pipe_adv1, 
    param_grid=param_grid_DecisionTreeRegressor, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_root_mean_squared_error")

grid_cv3.fit(X_train, Y_train) # Fit using training dataset

# Show GridSearch results for pipe_adv1 with cv = 5
res3 = grid_cv3.cv_results_
res3 = pd.DataFrame(res3)
res3 = res3.assign(experiment = 3)

# Subset to show columns needed for model evaluation
res3[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_model__max_depth', 'split0_test_score',
       'split1_test_score', 'split2_test_score', 'split3_test_score',
       'split4_test_score', 'mean_test_score',
       'std_test_score', 'rank_test_score', 'experiment']].sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__max_depth,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,experiment
0,0.016260,0.006208,0.008199,0.005818,1,-41.348469,-19.714660,-28.382780,-84.774285,-25.830226,-40.010084,23.468241,1,3
1,0.010011,0.000750,0.005044,0.000669,2,-90.922565,-33.289078,-27.220887,-84.006148,-25.151580,-52.118052,29.066237,2,3
3,0.010654,0.001379,0.005003,0.001164,6,-91.315415,-117.386628,-41.400689,-92.370609,-25.098210,-73.514310,34.561895,3,3
5,0.009435,0.000448,0.004257,0.000252,10,-91.835934,-121.308863,-42.730748,-91.272887,-26.021483,-74.633983,35.020840,4,3
4,0.009148,0.000392,0.004768,0.000352,8,-91.363010,-121.286034,-65.695993,-93.292494,-27.186725,-79.764851,31.635155,5,3
2,0.010771,0.000894,0.005105,0.000467,4,-90.608238,-116.828142,-27.868491,-87.039731,-85.847403,-81.638401,29.178311,6,3


In [19]:
# Pipeline 4: pipe_adv2
grid_cv4 = GridSearchCV(
    estimator=pipe_adv2, 
    param_grid=param_grid_DecisionTreeRegressor, 
    scoring = scoring, 
    cv = 5,
    refit = "neg_root_mean_squared_error")

grid_cv4.fit(X_train, Y_train) # Fit using training dataset

# Show GridSearch results for pipe_adv2 with cv = 5
res4 = grid_cv4.cv_results_
res4 = pd.DataFrame(res4)
res4 = res4.assign(experiment = 4)

# Subset to show columns needed for model evaluation
res4[['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_model__max_depth', 'split0_test_score',
       'split1_test_score', 'split2_test_score', 'split3_test_score',
       'split4_test_score', 'mean_test_score',
       'std_test_score', 'rank_test_score', 'experiment']].sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__max_depth,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,experiment
0,0.042084,0.013343,0.011210,0.006357,1,-41.348469,-19.714660,-28.382780,-84.774285,-25.830226,-40.010084,23.468241,1,4
1,0.039200,0.014732,0.007117,0.001422,2,-90.922565,-33.289078,-27.220887,-84.006148,-25.151580,-52.118052,29.066237,2,4
2,0.047091,0.022473,0.006567,0.001121,4,-90.077393,-86.029472,-27.868491,-86.706245,-24.918442,-63.120008,30.032928,3,4
3,0.030668,0.001069,0.006265,0.001313,6,-91.315415,-86.519278,-41.271965,-92.570609,-25.098210,-67.355095,28.436452,4,4
5,0.038573,0.006627,0.005072,0.000394,10,-91.700367,-120.670684,-43.085456,-91.140489,-25.583067,-74.436013,34.883263,5,4
4,0.045995,0.014988,0.010992,0.008856,8,-91.774647,-121.492585,-65.785770,-95.134174,-26.954791,-80.228393,31.952771,6,4


In [20]:
# Concatenate results from 4 models to evaluate best model
result_all = pd.concat([res1, res2, res3, res4])

result_all[['params', 'mean_test_score',
       'std_test_score', 'rank_test_score', 'experiment']].sort_values('mean_test_score', ascending = False).head(10).reset_index(drop=True)

,params,mean_test_score,std_test_score,rank_test_score,experiment
0,{'model__alpha': 3.0},-38.404070,24.694766,1,1
1,{'model__alpha': 2.0},-38.420702,24.513194,2,1
2,{'model__alpha': 3.0},-38.437365,24.681394,1,2
3,{'model__alpha': 4.0},-38.457334,24.727884,3,1
4,{'model__alpha': 2.0},-38.464825,24.501249,2,2
5,{'model__alpha': 4.0},-38.495408,24.702833,3,2
6,{'model__alpha': 5.0},-38.533590,24.703919,4,1
7,{'model__alpha': 5.0},-38.562629,24.684357,4,2
8,{'model__alpha': 1.0},-38.696527,24.192996,5,2
9,{'model__alpha': 1.0},-38.851315,24.125240,5,1


# Evaluate

+ Which model has the best performance?

# Export

+ Save the best performing model to a pickle file.

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.